# Model Prediksi Stok Website - Simple Linear Regression

Notebook ini mereplikasi model prediksi yang digunakan pada website StokManager.

Model utama:

`Y = a + bX`

Dengan definisi:
- `X` = konsumsi hari sebelumnya
- `Y` = konsumsi hari ini
- data konsumsi dihaluskan dengan EMA alpha = 0.05
- forecast stok dihitung iteratif: stok besok = stok hari ini - prediksi konsumsi besok

Catatan: model ini mengikuti `api/predict.py` pada website, bukan notebook/model lama yang memprediksi level stok langsung terhadap waktu.

## 1. Import Library dan Konfigurasi

Notebook ini memakai Python standar, pandas, dan matplotlib. Regresi linear dihitung manual agar rumus OLS terlihat jelas.

In [ ]:
from pathlib import Path
from datetime import datetime, timedelta
import json
import math

import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    display = print

pd.set_option('display.max_columns', 50)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True

MS_PER_DAY = 24 * 60 * 60 * 1000
CONSUMPTION_EMA_ALPHA = 0.05

# Jika notebook dijalankan dari folder scripts/, path pertama akan benar.
# Jika dijalankan dari root repo, fallback path kedua akan dipakai.
EXPORT_PATH = Path('../barcodescanesp32-default-rtdb-export.json')
if not EXPORT_PATH.exists():
    EXPORT_PATH = Path('barcodescanesp32-default-rtdb-export.json')

# Ubah ke barcode tertentu jika ingin memilih item manual.
# Biarkan None agar notebook otomatis memilih item paling berisiko dari data.
SELECTED_BARCODE = None

HORIZON_DAYS = 14
WEBSITE_HISTORY_DAYS = 30  # sama seperti grafik pada halaman /prediksi
TRAIN_RATIO = 0.85
TOP_N = 10

print('Export path:', EXPORT_PATH.resolve())

## 2. Fungsi Model

Bagian ini adalah versi notebook dari pipeline `api/predict.py`:
1. Rekonstruksi stok harian dari transaksi Firebase.
2. Hitung konsumsi harian.
3. Smoothing konsumsi memakai EMA.
4. Latih Simple Linear Regression dengan OLS manual.
5. Evaluasi MAE, RMSE, MAPE, dan R^2.
6. Forecast stok secara iteratif.

In [ ]:
def mean(values):
    return sum(values) / len(values) if values else 0.0


def day_key(timestamp_ms):
    return int(timestamp_ms // MS_PER_DAY) * MS_PER_DAY


def fmt_date(timestamp_ms):
    return datetime.fromtimestamp(timestamp_ms / 1000).strftime('%Y-%m-%d')


def build_daily_series(transactions, current_quantity):
    """Rekonstruksi level stok harian dari transaksi dan stok saat ini."""
    daily_delta = {}

    for tx in transactions:
        ts = int(tx.get('timestamp', 0) or 0)
        qty = int(tx.get('quantity', 0) or 0)
        tx_type = tx.get('type', 'out')
        key = day_key(ts)

        if tx_type == 'out':
            signed_qty = -abs(qty)
        elif tx_type == 'in':
            signed_qty = abs(qty)
        else:
            signed_qty = qty

        daily_delta[key] = daily_delta.get(key, 0) + signed_qty

    days = sorted(daily_delta)
    if not days:
        return []

    total_delta = sum(daily_delta.values())
    level = float(current_quantity) - total_delta

    series = []
    for day in days:
        level += daily_delta[day]
        series.append({'timestamp': day, 'quantity': max(0.0, level)})

    return series


def build_consumption_series(stock_series):
    sorted_series = sorted(stock_series, key=lambda point: point['timestamp'])
    consumption = []

    for i in range(1, len(sorted_series)):
        gap_days = (sorted_series[i]['timestamp'] - sorted_series[i - 1]['timestamp']) / MS_PER_DAY
        if gap_days <= 0:
            continue

        stock_delta = sorted_series[i - 1]['quantity'] - sorted_series[i]['quantity']
        daily_consumption = max(0.0, stock_delta / gap_days)
        consumption.append({
            'timestamp': sorted_series[i]['timestamp'],
            'consumption': daily_consumption,
        })

    return consumption


def smooth_consumption_series(series):
    if not series:
        return []

    smoothed = series[0]['consumption']
    result = []

    for index, point in enumerate(series):
        if index > 0:
            smoothed = (
                CONSUMPTION_EMA_ALPHA * point['consumption']
                + (1 - CONSUMPTION_EMA_ALPHA) * smoothed
            )

        result.append({
            'timestamp': point['timestamp'],
            'consumption': smoothed,
        })

    return result


def linear_regression(x, y):
    """OLS manual untuk Simple Linear Regression: y = intercept + slope * x."""
    n = len(x)
    if n == 0:
        return 0.0, 0.0
    if n == 1:
        return y[0], 0.0

    mx = mean(x)
    my = mean(y)
    numerator = 0.0
    denominator = 0.0

    for i in range(n):
        dx = x[i] - mx
        numerator += dx * (y[i] - my)
        denominator += dx * dx

    if denominator == 0:
        return my, 0.0

    slope = numerator / denominator
    intercept = my - slope * mx
    return intercept, slope


def fit_consumption_regression(stock_series):
    stock_series = sorted(stock_series, key=lambda point: point['timestamp'])
    raw_consumption_series = build_consumption_series(stock_series)
    consumption_series = smooth_consumption_series(raw_consumption_series)

    x = []
    y = []
    for i in range(1, len(consumption_series)):
        x.append(consumption_series[i - 1]['consumption'])
        y.append(consumption_series[i]['consumption'])

    fallback_consumption = raw_consumption_series[-1]['consumption'] if raw_consumption_series else 0.0
    if y:
        intercept, consumption_slope = linear_regression(x, y)
    else:
        intercept, consumption_slope = fallback_consumption, 0.0

    avg_daily = mean([point['consumption'] for point in raw_consumption_series])

    return {
        'type': 'Simple Linear Regression (daily consumption)',
        'intercept': intercept,
        'slope': -avg_daily,
        'consumptionIntercept': intercept,
        'consumptionSlope': consumption_slope,
        'avgDailyConsumption': avg_daily,
        'n': max(1, len(consumption_series)),
        'lastConsumption': consumption_series[-1]['consumption'] if consumption_series else fallback_consumption,
        # field tambahan untuk visualisasi notebook
        'trainX': x,
        'trainY': y,
        'rawConsumptionSeries': raw_consumption_series,
        'smoothedConsumptionSeries': consumption_series,
    }


def predict_next_consumption(model, previous_consumption):
    predicted = model['consumptionIntercept'] + model['consumptionSlope'] * previous_consumption
    return max(0.0, predicted)


def calculate_metrics(actual, predicted):
    if not actual or not predicted:
        return {'mae': 0.0, 'rmse': 0.0, 'mape': 0.0, 'r2': 0.0}

    errors = [actual[i] - predicted[i] for i in range(len(actual))]
    mae = mean([abs(error) for error in errors])
    rmse = math.sqrt(mean([error ** 2 for error in errors]))
    non_zero_percentage_errors = [
        abs((actual[i] - predicted[i]) / actual[i]) * 100
        for i in range(len(actual))
        if actual[i] != 0
    ]
    mape = mean(non_zero_percentage_errors)

    avg_actual = mean(actual)
    ss_res = sum(error ** 2 for error in errors)
    ss_tot = sum((value - avg_actual) ** 2 for value in actual)
    r2 = 1.0 if ss_tot == 0 and ss_res == 0 else 0.0 if ss_tot == 0 else 1 - ss_res / ss_tot

    return {'mae': mae, 'rmse': rmse, 'mape': mape, 'r2': r2}


def evaluate_consumption(model, stock_series):
    consumption_series = smooth_consumption_series(build_consumption_series(stock_series))
    if not consumption_series:
        return {'mae': 0.0, 'rmse': 0.0, 'mape': 0.0, 'r2': 0.0}

    actual = []
    predicted = []
    for i in range(1, len(consumption_series)):
        actual.append(consumption_series[i]['consumption'])
        predicted.append(predict_next_consumption(model, consumption_series[i - 1]['consumption']))

    return calculate_metrics(actual, predicted)


def estimate_stockout_date(model, current_quantity, base_timestamp):
    base_date = datetime.fromtimestamp(base_timestamp / 1000)
    if current_quantity <= 0:
        return base_date.strftime('%Y-%m-%d')

    quantity = float(current_quantity)
    previous_consumption = model.get('lastConsumption', model['avgDailyConsumption'])

    for day in range(1, 3651):
        consumption = predict_next_consumption(model, previous_consumption)
        if consumption <= 0 and model['avgDailyConsumption'] <= 0:
            return None

        quantity = max(0.0, quantity - consumption)
        previous_consumption = consumption
        if quantity <= 0:
            return (base_date + timedelta(days=day)).strftime('%Y-%m-%d')

    return None


def predict_stock(stock_series, horizon_days=14, train_ratio=0.85):
    stock_series = sorted(stock_series, key=lambda point: point['timestamp'])
    if len(stock_series) < 2:
        return {'error': 'Not enough data'}

    split_idx = min(len(stock_series), max(2, int(len(stock_series) * train_ratio)))
    train = stock_series[:split_idx]
    test = stock_series[split_idx:]

    model = fit_consumption_regression(train)
    metrics = evaluate_consumption(model, stock_series)

    last_qty = stock_series[-1]['quantity']
    last_ts = stock_series[-1]['timestamp']
    current_qty = float(last_qty)
    previous_consumption = model.get('lastConsumption', model['avgDailyConsumption'])
    forecast = []

    for day in range(1, horizon_days + 1):
        ts = last_ts + day * MS_PER_DAY
        predicted_consumption = predict_next_consumption(model, previous_consumption)
        current_qty = max(0.0, current_qty - predicted_consumption)
        previous_consumption = predicted_consumption
        forecast.append({
            'timestamp': int(ts),
            'predictedQuantity': round(current_qty, 1),
            'estimatedConsumption': round(predicted_consumption, 1),
        })

    return {
        'source': 'lr-consumption-notebook',
        'model': model,
        'metrics': metrics,
        'forecast': forecast,
        'stockoutDate': estimate_stockout_date(model, last_qty, last_ts),
        'nTrain': max(1, len(train) - 1),
        'nTest': len(test),
    }

## 3. Load Data Firebase Export

File export Firebase berisi `inventory` dan `transactions`. Website tidak menyimpan histori stok penuh, sehingga level stok harian direkonstruksi dari transaksi dan stok saat ini.

In [ ]:
def load_firebase_export(path):
    if not path.exists():
        raise FileNotFoundError(f'File export Firebase tidak ditemukan: {path}')

    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


def collect_item_datasets(firebase_data):
    inventory = firebase_data.get('inventory') or {}
    transactions = firebase_data.get('transactions') or {}

    tx_by_barcode = {}
    for tx in transactions.values():
        barcode = tx.get('productBarcode') or tx.get('barcode') or ''
        if not barcode:
            continue

        tx_by_barcode.setdefault(barcode, []).append({
            'timestamp': int(tx.get('timestamp', 0) or 0),
            'quantity': int(tx.get('quantity', 0) or 0),
            'type': tx.get('type', 'out'),
            'productName': tx.get('productName', ''),
        })

    datasets = []
    for item in inventory.values():
        barcode = item.get('barcode') or ''
        item_transactions = tx_by_barcode.get(barcode, [])
        if len(item_transactions) < 2:
            continue

        current_quantity = float(item.get('quantity', 0) or 0)
        series = build_daily_series(item_transactions, current_quantity)
        if len(series) < 2:
            continue

        datasets.append({
            'id': item.get('id'),
            'barcode': barcode,
            'name': item.get('name', ''),
            'category': item.get('category', ''),
            'currentQuantity': current_quantity,
            'minStock': float(item.get('minStock', 0) or 0),
            'transactions': item_transactions,
            'series': series,
        })

    return datasets


firebase_data = load_firebase_export(EXPORT_PATH)
datasets = collect_item_datasets(firebase_data)

print('Jumlah item inventory     :', len(firebase_data.get('inventory') or {}))
print('Jumlah transaksi          :', len(firebase_data.get('transactions') or {}))
print('Item siap diprediksi      :', len(datasets))

overview = pd.DataFrame([
    {
        'barcode': item['barcode'],
        'name': item['name'],
        'category': item['category'],
        'current_qty': item['currentQuantity'],
        'min_stock': item['minStock'],
        'transactions': len(item['transactions']),
        'daily_points': len(item['series']),
        'start_date': fmt_date(item['series'][0]['timestamp']),
        'end_date': fmt_date(item['series'][-1]['timestamp']),
    }
    for item in datasets
])

display(overview.sort_values(['daily_points', 'transactions'], ascending=False).head(20))

## 4. Jalankan Model untuk Semua Item

Tabel berikut membantu memilih item yang paling menarik untuk ditampilkan saat seminar: ada parameter model, metrik evaluasi, dan estimasi risiko stok habis.

In [ ]:
rows = []

for item in datasets:
    result = predict_stock(item['series'], horizon_days=HORIZON_DAYS, train_ratio=TRAIN_RATIO)
    if 'error' in result:
        continue

    forecast = result['forecast']
    predicted_lowest = min(point['predictedQuantity'] for point in forecast) if forecast else None
    days_to_stockout = None
    for index, point in enumerate(forecast):
        if point['predictedQuantity'] <= 0:
            days_to_stockout = index + 1
            break

    if days_to_stockout is None:
        avg_daily = result['model']['avgDailyConsumption']
        if avg_daily > 0 and item['currentQuantity'] > 0:
            days_to_stockout = round(item['currentQuantity'] / avg_daily)

    rows.append({
        'barcode': item['barcode'],
        'name': item['name'],
        'current_qty': item['currentQuantity'],
        'min_stock': item['minStock'],
        'daily_points': len(item['series']),
        'a_intercept': result['model']['consumptionIntercept'],
        'b_slope': result['model']['consumptionSlope'],
        'avg_consumption': result['model']['avgDailyConsumption'],
        'stock_slope_per_day': result['model']['slope'],
        'mae': result['metrics']['mae'],
        'rmse': result['metrics']['rmse'],
        'mape': result['metrics']['mape'],
        'r2': result['metrics']['r2'],
        'predicted_lowest_14d': predicted_lowest,
        'days_to_stockout': days_to_stockout,
        'stockout_date': result['stockoutDate'],
    })

summary = pd.DataFrame(rows)
summary['stockout_sort'] = summary['days_to_stockout'].fillna(999999)
summary = summary.sort_values(['stockout_sort', 'predicted_lowest_14d', 'daily_points'], ascending=[True, True, False])

display(summary.drop(columns=['stockout_sort']).head(TOP_N).round(3))

## 5. Detail Satu Item

Jika ingin memilih item manual, ubah `SELECTED_BARCODE` pada bagian konfigurasi. Jika `None`, notebook memilih item teratas dari tabel risiko.

In [ ]:
if summary.empty:
    raise ValueError('Tidak ada item yang dapat diprediksi. Periksa data transaksi.')

selected_barcode = SELECTED_BARCODE or summary.iloc[0]['barcode']
selected = next(item for item in datasets if item['barcode'] == selected_barcode)
result = predict_stock(selected['series'], horizon_days=HORIZON_DAYS, train_ratio=TRAIN_RATIO)
model = result['model']

print('Item              :', selected['name'])
print('Barcode           :', selected['barcode'])
print('Stok saat ini     :', selected['currentQuantity'])
print('Minimum stok      :', selected['minStock'])
print('Jumlah data harian:', len(selected['series']))
print('Rentang data      :', fmt_date(selected['series'][0]['timestamp']), 's/d', fmt_date(selected['series'][-1]['timestamp']))
print()
print('Persamaan regresi konsumsi:')
print(f"Y = {model['consumptionIntercept']:.4f} + {model['consumptionSlope']:.4f} X")
print('X = konsumsi hari sebelumnya')
print('Y = konsumsi hari ini')
print()
print('Rata-rata konsumsi harian :', round(model['avgDailyConsumption'], 3))
print('Slope stok untuk UI       :', round(model['slope'], 3), 'unit/hari')
print('MAE                       :', round(result['metrics']['mae'], 3))
print('RMSE                      :', round(result['metrics']['rmse'], 3))
print('MAPE                      :', round(result['metrics']['mape'], 3), '%')
print('R^2                       :', round(result['metrics']['r2'], 3))
print('n train                   :', result['nTrain'])
print('n test                    :', result['nTest'])
print('Estimasi stockout         :', result['stockoutDate'] or 'Tidak terprediksi')

## 6. Tabel Forecast 14 Hari

Forecast stok tidak berupa garis lurus paksa. Model memprediksi konsumsi harian, lalu stok dikurangi dari hari ke hari.

In [ ]:
forecast_df = pd.DataFrame(result['forecast'])
forecast_df['date'] = forecast_df['timestamp'].apply(fmt_date)
forecast_df = forecast_df[['date', 'estimatedConsumption', 'predictedQuantity']]
forecast_df.columns = ['tanggal', 'prediksi_konsumsi', 'prediksi_stok']

display(forecast_df)

## 7. Visualisasi Hasil

Grafik pertama dibuat mengikuti tampilan website: historis dibatasi 30 hari terakhir, forecast 14 hari, dan sumbu X memakai urutan titik seperti komponen SVG `PredictionChart`. Grafik kedua menunjukkan konsumsi harian sebelum dan sesudah EMA.

In [ ]:
history_df = pd.DataFrame(selected['series'])
history_df['date'] = pd.to_datetime(history_df['timestamp'], unit='ms')

# Website membatasi historis pada grafik menjadi 30 hari terakhir untuk readability.
history_cutoff = int(datetime.now().timestamp() * 1000) - WEBSITE_HISTORY_DAYS * MS_PER_DAY
history_plot_df = history_df[history_df['timestamp'] >= history_cutoff].copy()
if history_plot_df.empty:
    # Fallback jika export lama: tetap tampilkan 30 titik terakhir agar grafik tidak kosong.
    history_plot_df = history_df.tail(WEBSITE_HISTORY_DAYS).copy()

forecast_plot_df = pd.DataFrame(result['forecast'])
forecast_plot_df['date'] = pd.to_datetime(forecast_plot_df['timestamp'], unit='ms')

raw_consumption_df = pd.DataFrame(model['rawConsumptionSeries'])
smooth_consumption_df = pd.DataFrame(model['smoothedConsumptionSeries'])
if not raw_consumption_df.empty:
    raw_consumption_df['date'] = pd.to_datetime(raw_consumption_df['timestamp'], unit='ms')
if not smooth_consumption_df.empty:
    smooth_consumption_df['date'] = pd.to_datetime(smooth_consumption_df['timestamp'], unit='ms')

website_chart_df = pd.concat([
    pd.DataFrame({
        'date': history_plot_df['date'].dt.strftime('%d %b'),
        'actual': history_plot_df['quantity'],
        'predicted': float('nan'),
    }),
    pd.DataFrame({
        'date': forecast_plot_df['date'].dt.strftime('%d %b'),
        'actual': float('nan'),
        'predicted': forecast_plot_df['predictedQuantity'],
    }),
], ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

x_axis = list(range(len(website_chart_df)))
axes[0].plot(x_axis, website_chart_df['actual'], marker='o', label='Stok historis')
axes[0].plot(x_axis, website_chart_df['predicted'], marker='o', linestyle='--', label='Forecast 14 hari')
axes[0].axhline(selected['minStock'], color='red', linestyle=':', label='Minimum stok')
axes[0].set_title('Historis dan Forecast Stok (mode website)')
axes[0].set_xlabel('Tanggal')
axes[0].set_ylabel('Jumlah stok')
axes[0].legend()
label_step = max(1, math.ceil(len(website_chart_df) / 8))
label_positions = [i for i in range(len(website_chart_df)) if i % label_step == 0 or i == len(website_chart_df) - 1]
axes[0].set_xticks(label_positions)
axes[0].set_xticklabels(website_chart_df.loc[label_positions, 'date'], rotation=30)

if not raw_consumption_df.empty:
    axes[1].plot(raw_consumption_df['date'], raw_consumption_df['consumption'], marker='o', alpha=0.45, label='Konsumsi raw')
if not smooth_consumption_df.empty:
    axes[1].plot(smooth_consumption_df['date'], smooth_consumption_df['consumption'], marker='o', label='Konsumsi EMA')
axes[1].set_title('Konsumsi Harian dan EMA')
axes[1].set_xlabel('Tanggal')
axes[1].set_ylabel('Konsumsi per hari')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle(selected['name'])
plt.tight_layout()
plt.show()

## 8. Plot Regresi Linear Sederhana

Plot ini menunjukkan pasangan data training `(X, Y)` dan garis regresi `Y = a + bX`.

In [ ]:
x = model['trainX']
y = model['trainY']

if len(x) == 0:
    print('Data training konsumsi belum cukup untuk scatter regresi. Model memakai fallback konsumsi terakhir.')
else:
    min_x = min(x)
    max_x = max(x)
    if min_x == max_x:
        line_x = [min_x - 0.5, max_x + 0.5]
    else:
        line_x = [min_x, max_x]
    line_y = [model['consumptionIntercept'] + model['consumptionSlope'] * value for value in line_x]

    plt.figure(figsize=(8, 6))
    plt.scatter(x, y, label='Data training', s=60)
    plt.plot(line_x, line_y, color='red', label='Garis regresi')
    plt.title('Simple Linear Regression Konsumsi Harian')
    plt.xlabel('X = konsumsi hari sebelumnya')
    plt.ylabel('Y = konsumsi hari ini')
    plt.legend()
    plt.show()

    print(f"Persamaan: Y = {model['consumptionIntercept']:.4f} + {model['consumptionSlope']:.4f} X")

## 9. Ringkasan untuk Slide/Seminar

Poin yang bisa dijelaskan:
- Data berasal dari Firebase Realtime Database: `inventory` dan `transactions`.
- Histori level stok direkonstruksi dari stok saat ini dan transaksi harian.
- Model menggunakan Simple Linear Regression biasa: `Y = a + bX`.
- Variabel `X` adalah konsumsi hari sebelumnya, variabel `Y` adalah konsumsi hari ini.
- Forecast stok dihitung iteratif dari prediksi konsumsi, sehingga grafik forecast tidak harus lurus.
- Metrik evaluasi yang ditampilkan: MAE, RMSE, MAPE, dan R^2.